In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/databricks-hackathon-backend/Virtue Foundation Ghana v0.3 - Sheet1.csv"))

In [0]:
CATALOG="workspace"
SCHEMA="default"
TABLE="databricks_hackathon_backend"
VOLUME_PATH=f"/Volumes/{CATALOG}/{SCHEMA}/databricks-hackathon-backend/Virtue Foundation Ghana v0.3 - Sheet1.csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
df = (spark.read
      .option("header", "true")        # first row is column names
      .option("inferSchema", "true")   # auto-detect int/string/bool
      .option("encoding", "UTF-8")     # force UTF-8; use latin1 if errors
      .option("multiLine", "true")     # handles newlines inside text fields
      .option("escape", '"')           # handles quoted commas
      .csv(VOLUME_PATH))

display(df.limit(10))
print(f"Row count: {df.count()}")
print(f"Columns:   {df.columns}")

In [0]:
# ── Cell 4: Write as a managed Delta Lake table ────────────────────
# Sanitize column names by replacing spaces with underscores
df_clean = df.withColumnRenamed("mongo DB", "mongo_DB")

(df_clean.write
   .format("delta")
   .mode("overwrite")                  # safe to re-run; replaces existing table
   .option("overwriteSchema", "true")  # allows schema changes on re-run
   .saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}"))

print(f"Table written: {CATALOG}.{SCHEMA}.{TABLE}")

In [0]:
# ── Cell 5: Verify it's registered and queryable ──────────────────
result = spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.{TABLE} LIMIT 5")
display(result)

# Check row count matches
count = spark.sql(f"SELECT COUNT(*) as total FROM {CATALOG}.{SCHEMA}.{TABLE}")
display(count)

# Confirm it is a Delta table
spark.sql(f"DESCRIBE DETAIL {CATALOG}.{SCHEMA}.{TABLE}").select("format", "location", "numFiles").show()
# Expected output under 'format': delta